In [ ]:
import os
os.chdir(os.environ.get("PROJECT_ROOT", "../.."))

import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Configuration
results_base = Path("results/expert-utilization")
num_experts = 32

models = {
    "290m-seed3407-32e": "Baseline",
    "290m-dro-activation-beta0.999-eta0.001-alpha1.0-norml2_fine_grained_32e": "DRMoET"
}

checkpoints = [540, 2160, 3780, 5400, 7020, 8640, 10260, 12420, 14040, 16000]
layers = [2, 3, 4, 5, 6, 7, 8, 9]

# Paper style
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'legend.fontsize': 11,
    'figure.dpi': 150,
    'savefig.dpi': 300,
})

output_dir = Path("figures/expert-utilization-32e")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Results base: {results_base}")
print(f"Models: {list(models.keys())}")
print(f"Checkpoints: {checkpoints}")
print(f"Layers: {layers}")

In [ ]:
def load_layer_data(model_name: str, checkpoint: int, layer: int) -> dict:
    """Load per-layer results from JSON."""
    path = results_base / model_name / str(checkpoint) / f"layer_{layer}.json"
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return None


def load_aggregate_data(model_name: str, checkpoint: int) -> dict:
    """Load aggregate results from JSON."""
    path = results_base / model_name / str(checkpoint) / "aggregate.json"
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return None


# Load all data
all_data = {}
for model_name, label in models.items():
    print(f"Loading {label}...")
    all_data[model_name] = {}
    
    for ckpt in checkpoints:
        all_data[model_name][ckpt] = {
            'aggregate': load_aggregate_data(model_name, ckpt),
            'layers': {layer: load_layer_data(model_name, ckpt, layer) for layer in layers}
        }

# Check what data is available
for model_name, label in models.items():
    available = sum(1 for ckpt in checkpoints if all_data[model_name][ckpt]['aggregate'] is not None)
    print(f"{label}: {available}/{len(checkpoints)} checkpoints with aggregate data")

In [ ]:
# Extract CV values over training
stats = {}
for model_name, label in models.items():
    stats[model_name] = {
        'checkpoints': [],
        'cv': [],
        'gini': [],
        'entropy_ratio': [],
        'per_layer_cv': {layer: [] for layer in layers}
    }
    
    for ckpt in checkpoints:
        agg = all_data[model_name][ckpt]['aggregate']
        if agg:
            stats[model_name]['checkpoints'].append(ckpt)
            stats[model_name]['cv'].append(agg['cv_percent'])
            stats[model_name]['gini'].append(agg['gini'])
            stats[model_name]['entropy_ratio'].append(agg.get('entropy_ratio', 0))
            
            # Per-layer CV
            for layer in layers:
                layer_data = all_data[model_name][ckpt]['layers'][layer]
                if layer_data:
                    stats[model_name]['per_layer_cv'][layer].append(layer_data['cv_percent'])

# Print summary table
baseline_name = list(models.keys())[0]
dro_name = list(models.keys())[1]

print("\n" + "="*80)
print(f"{'Checkpoint':>10} | {'Baseline CV':>12} | {'DRMoET CV':>12} | {'Diff':>10} | {'Winner':>10}")
print("-"*80)

for i, ckpt in enumerate(stats[baseline_name]['checkpoints']):
    b_cv = stats[baseline_name]['cv'][i]
    dro_idx = stats[dro_name]['checkpoints'].index(ckpt) if ckpt in stats[dro_name]['checkpoints'] else None
    if dro_idx is not None:
        d_cv = stats[dro_name]['cv'][dro_idx]
        diff = d_cv - b_cv
        winner = "Baseline" if b_cv < d_cv else "DRMoET"
        print(f"{ckpt:>10} | {b_cv:>11.2f}% | {d_cv:>11.2f}% | {diff:>+9.2f}% | {winner:>10}")
    else:
        print(f"{ckpt:>10} | {b_cv:>11.2f}% | {'N/A':>12} | {'N/A':>10} | {'N/A':>10}")

print("-"*80)
if stats[baseline_name]['cv'] and stats[dro_name]['cv']:
    print(f"{'Average':>10} | {np.mean(stats[baseline_name]['cv']):>11.2f}% | {np.mean(stats[dro_name]['cv']):>11.2f}% |")
print("="*80)

In [ ]:
# Plot: Per-Layer CV Over Training (like router saturation)
# Shows how expert utilization changes over checkpoints for each layer

from matplotlib.ticker import FuncFormatter

def pct_format(x, pos):
    return f"{x:.0f}%"

# Normalize checkpoints to training progress (0-100%)
steps = np.array(checkpoints) / checkpoints[-1]

# 2x4 grid: each subplot shows one layer, both models compared
fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=True)
axes = axes.flatten()

colors = ['#2E86AB', '#A23B72']
markers = ['o', 's']
model_names = list(models.keys())
model_labels = list(models.values())

for idx, layer in enumerate(layers):
    ax = axes[idx]
    
    for midx, (model_name, label) in enumerate(models.items()):
        cv_values = stats[model_name]['per_layer_cv'][layer]
        if cv_values:
            ax.plot(steps * 100, cv_values, marker=markers[midx], color=colors[midx], 
                    label=label, linewidth=1.5, markersize=5)
    
    ax.set_title(f'Layer {layer}', fontsize=12)
    ax.tick_params(axis='both', labelsize=10)
    ax.xaxis.set_major_formatter(FuncFormatter(pct_format))
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, None)

# Add legend to last subplot
axes[-1].legend(loc='upper right', fontsize=9, frameon=True)

# Common labels
fig.text(0.5, 0.02, 'Training Progress', ha='center', fontsize=14)
fig.text(0.02, 0.5, 'CV (%) - higher = more varied utilization', va='center', rotation='vertical', fontsize=14)
fig.suptitle('Expert Utilization CV Over Training by Layer', fontsize=14, y=0.98)

plt.tight_layout(rect=[0.03, 0.03, 1, 0.96])
plt.savefig(output_dir / 'cv_over_training_per_layer.pdf', bbox_inches='tight')
plt.savefig(output_dir / 'cv_over_training_per_layer.png', bbox_inches='tight')
plt.show()
print(f"Saved: {output_dir / 'cv_over_training_per_layer.pdf'}")

In [ ]:
# Stacked: All layers for each model (like router saturation format)
fig, axes = plt.subplots(2, 1, figsize=(10, 10), sharex=True, sharey=True)

for ax, (model_name, label) in zip(axes, models.items()):
    for layer in layers:
        cv_values = stats[model_name]['per_layer_cv'][layer]
        if cv_values:
            ax.plot(steps * 100, cv_values, marker='o', label=f'L{layer}', linewidth=1.5, markersize=4)
    
    ax.set_title(f'{label}', fontsize=14)
    ax.set_ylabel('CV (%)', fontsize=12)
    ax.tick_params(axis='both', labelsize=11)
    ax.xaxis.set_major_formatter(FuncFormatter(pct_format))
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', fontsize=9, frameon=False, ncol=4)

axes[-1].set_xlabel('Training Progress', fontsize=12)

plt.suptitle('Expert Utilization CV Over Training', fontsize=14)
plt.tight_layout()
plt.savefig(output_dir / 'cv_over_training_side_by_side.pdf', bbox_inches='tight')
plt.savefig(output_dir / 'cv_over_training_side_by_side.png', bbox_inches='tight')
plt.show()
print(f"Saved: {output_dir / 'cv_over_training_side_by_side.pdf'}")

In [ ]:
# Heatmap: CV by checkpoint x layer for both models
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, (model_name, label) in zip(axes, models.items()):
    # Build matrix: checkpoints x layers
    matrix = np.zeros((len(checkpoints), len(layers)))
    for i, ckpt in enumerate(checkpoints):
        for j, layer in enumerate(layers):
            layer_data = all_data[model_name][ckpt]['layers'][layer]
            if layer_data:
                matrix[i, j] = layer_data['cv_percent']
    
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn_r')  # Red=high CV, Green=low CV
    ax.set_xlabel('Layer')
    ax.set_ylabel('Checkpoint')
    ax.set_xticks(range(len(layers)))
    ax.set_xticklabels([f'L{l}' for l in layers])
    ax.set_yticks(range(len(checkpoints)))
    ax.set_yticklabels([f'{int(c/1000)}k' for c in checkpoints])
    ax.set_title(f'{label}')
    plt.colorbar(im, ax=ax, label='CV (%)')

plt.suptitle('Expert Utilization CV: Checkpoint x Layer', fontsize=14)
plt.tight_layout()
plt.savefig(output_dir / 'cv_heatmap_checkpoint_layer.pdf', bbox_inches='tight')
plt.savefig(output_dir / 'cv_heatmap_checkpoint_layer.png', bbox_inches='tight')
plt.show()
print(f"Saved: {output_dir / 'cv_heatmap_checkpoint_layer.pdf'}")

In [ ]:
# Direct Expert Utilization: Token percentage per expert per layer
# Shows actual distribution, not just CV summary

# Plot: 2x4 grid - each layer shows bar chart of expert usage %
final_ckpt = stats[baseline_name]['checkpoints'][-1] if stats[baseline_name]['checkpoints'] else checkpoints[-1]

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for idx, layer in enumerate(layers):
    ax = axes[idx]
    x = np.arange(num_experts)
    width = 0.4
    
    # Get counts and convert to percentage
    baseline_data = all_data[baseline_name][final_ckpt]['layers'][layer]
    dro_data = all_data[dro_name][final_ckpt]['layers'][layer]
    
    if baseline_data:
        baseline_counts = np.array(baseline_data['counts'])
        baseline_pct = baseline_counts / baseline_counts.sum() * 100
        ax.bar(x - width/2, baseline_pct, width, label='Baseline', color='#2E86AB', alpha=0.7)
    
    if dro_data:
        dro_counts = np.array(dro_data['counts'])
        dro_pct = dro_counts / dro_counts.sum() * 100
        ax.bar(x + width/2, dro_pct, width, label='DRMoET', color='#A23B72', alpha=0.7)
    
    # Uniform line
    uniform = 100 / num_experts
    ax.axhline(uniform, color='gray', linestyle='--', alpha=0.7, linewidth=1.5)
    
    ax.set_xlabel('Expert ID')
    ax.set_ylabel('Token Share (%)')
    ax.set_title(f'Layer {layer}')
    ax.set_ylim(0, max(baseline_pct.max(), dro_pct.max()) * 1.1 if baseline_data and dro_data else 10)
    if idx == 0:
        ax.legend(loc='upper right', fontsize=9)

plt.suptitle(f'Expert Utilization by Layer (Step {final_ckpt}) - Uniform = {100/num_experts:.2f}%', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(output_dir / 'expert_utilization_per_layer.pdf', bbox_inches='tight')
plt.savefig(output_dir / 'expert_utilization_per_layer.png', bbox_inches='tight')
plt.show()
print(f"Saved: {output_dir / 'expert_utilization_per_layer.pdf'}")

In [ ]:
# Heatmap: Expert Utilization % (layers x experts) for both models
# Cleaner visualization showing actual percentages

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

final_ckpt = stats[baseline_name]['checkpoints'][-1] if stats[baseline_name]['checkpoints'] else checkpoints[-1]
uniform = 100 / num_experts

for idx, (model_name, label) in enumerate(models.items()):
    # Build matrix: layers x experts (percentage)
    matrix = np.zeros((len(layers), num_experts))
    for i, layer in enumerate(layers):
        layer_data = all_data[model_name][final_ckpt]['layers'][layer]
        if layer_data:
            counts = np.array(layer_data['counts'])
            matrix[i] = counts / counts.sum() * 100
    
    ax = axes[idx]
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=matrix.max())
    
    ax.set_xlabel('Expert ID')
    ax.set_ylabel('Layer')
    ax.set_xticks(range(0, num_experts, 2))
    ax.set_yticks(range(len(layers)))
    ax.set_yticklabels([f'L{l}' for l in layers])
    ax.set_title(f'{label} (uniform = {uniform:.2f}%)')
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Token Share (%)')

plt.suptitle(f'Expert Utilization Heatmap (Step {final_ckpt})', fontsize=14)
plt.tight_layout()
plt.savefig(output_dir / 'expert_utilization_heatmap.pdf', bbox_inches='tight')
plt.savefig(output_dir / 'expert_utilization_heatmap.png', bbox_inches='tight')
plt.show()
print(f"Saved: {output_dir / 'expert_utilization_heatmap.pdf'}")

In [ ]:
# Print summary: Top/Bottom experts per layer
final_ckpt = stats[baseline_name]['checkpoints'][-1] if stats[baseline_name]['checkpoints'] else checkpoints[-1]
uniform = 100 / num_experts

print(f"Expert Utilization Summary (Step {final_ckpt})")
print(f"Uniform distribution = {uniform:.2f}%")
print("="*80)

for model_name, label in models.items():
    print(f"\n{label}:")
    print("-"*60)
    print(f"{'Layer':<8} {'Most Used Expert':<25} {'Least Used Expert':<25}")
    print("-"*60)
    
    for layer in layers:
        layer_data = all_data[model_name][final_ckpt]['layers'][layer]
        if layer_data:
            counts = np.array(layer_data['counts'])
            pct = counts / counts.sum() * 100
            
            max_exp = pct.argmax()
            min_exp = pct.argmin()
            
            print(f"L{layer:<7} Expert {max_exp:2d}: {pct[max_exp]:5.2f}%        Expert {min_exp:2d}: {pct[min_exp]:5.2f}%")
    
    # Aggregate
    agg = all_data[model_name][final_ckpt]['aggregate']
    if agg:
        counts = np.array(agg['counts'])
        pct = counts / counts.sum() * 100
        print("-"*60)
        print(f"{'Total':<8} Expert {pct.argmax():2d}: {pct.max():5.2f}%        Expert {pct.argmin():2d}: {pct.min():5.2f}%")
        print(f"         Range: {pct.min():.2f}% - {pct.max():.2f}% (uniform={uniform:.2f}%)")

In [ ]:
# Difference plot: DRMoET - Baseline utilization per expert
# Positive = DRMoET uses more, Negative = Baseline uses more

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

final_ckpt = stats[baseline_name]['checkpoints'][-1] if stats[baseline_name]['checkpoints'] else checkpoints[-1]

for idx, layer in enumerate(layers):
    ax = axes[idx]
    
    baseline_data = all_data[baseline_name][final_ckpt]['layers'][layer]
    dro_data = all_data[dro_name][final_ckpt]['layers'][layer]
    
    if baseline_data and dro_data:
        baseline_pct = np.array(baseline_data['counts']) / sum(baseline_data['counts']) * 100
        dro_pct = np.array(dro_data['counts']) / sum(dro_data['counts']) * 100
        diff = dro_pct - baseline_pct
        
        colors = ['#A23B72' if d > 0 else '#2E86AB' for d in diff]
        ax.bar(range(num_experts), diff, color=colors, alpha=0.8)
        ax.axhline(0, color='black', linewidth=1)
    
    ax.set_xlabel('Expert ID')
    ax.set_ylabel('Difference (%)')
    ax.set_title(f'Layer {layer}')

plt.suptitle(f'Expert Utilization Difference: DRMoET - Baseline (Step {final_ckpt})\nPink = DRMoET uses more, Blue = Baseline uses more', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(output_dir / 'expert_utilization_diff.pdf', bbox_inches='tight')
plt.savefig(output_dir / 'expert_utilization_diff.png', bbox_inches='tight')
plt.show()
print(f"Saved: {output_dir / 'expert_utilization_diff.pdf'}")

In [ ]:
# Plot 1: CV Over Training
fig, ax = plt.subplots(figsize=(10, 6))

colors = {baseline_name: '#2E86AB', dro_name: '#A23B72'}

for model_name, label in models.items():
    if stats[model_name]['checkpoints']:
        ax.plot(stats[model_name]['checkpoints'], stats[model_name]['cv'], 
                'o-', label=label, color=colors[model_name], linewidth=2, markersize=8)

ax.set_xlabel('Training Step')
ax.set_ylabel('Coefficient of Variation (%) - lower is more uniform')
ax.set_title('Expert Load Imbalance Over Training')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'cv_over_training.pdf')
plt.savefig(output_dir / 'cv_over_training.png')
plt.show()

In [ ]:
# Plot 2: Per-Layer CV Comparison at final checkpoint
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(layers))
width = 0.35

baseline_layer_cv = [stats[baseline_name]['per_layer_cv'][l][-1] if stats[baseline_name]['per_layer_cv'][l] else 0 for l in layers]
dro_layer_cv = [stats[dro_name]['per_layer_cv'][l][-1] if stats[dro_name]['per_layer_cv'][l] else 0 for l in layers]

ax.bar(x - width/2, baseline_layer_cv, width, label='Baseline', color='#2E86AB', alpha=0.8)
ax.bar(x + width/2, dro_layer_cv, width, label='DRMoET', color='#A23B72', alpha=0.8)

ax.set_xlabel('Layer')
ax.set_ylabel('CV (%) - lower is more uniform')
final_ckpt = stats[baseline_name]['checkpoints'][-1] if stats[baseline_name]['checkpoints'] else 'N/A'
ax.set_title(f'Per-Layer Expert Load Imbalance (Step {final_ckpt})')
ax.set_xticks(x)
ax.set_xticklabels([f'Layer {l}' for l in layers])
ax.legend()
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(output_dir / 'per_layer_cv.pdf')
plt.savefig(output_dir / 'per_layer_cv.png')
plt.show()

In [ ]:
# Plot 3: Expert Distribution Heatmaps (2 models stacked)
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

final_ckpt = stats[baseline_name]['checkpoints'][-1] if stats[baseline_name]['checkpoints'] else checkpoints[-1]

for idx, (model_name, label) in enumerate(models.items()):
    matrix = np.zeros((len(layers), num_experts))
    for i, layer in enumerate(layers):
        layer_data = all_data[model_name][final_ckpt]['layers'][layer]
        if layer_data:
            counts = np.array(layer_data['counts'])
            matrix[i] = counts / (counts.sum() + 1e-10) * 100
    
    im = axes[idx].imshow(matrix, aspect='auto', cmap='YlOrRd')
    axes[idx].set_xlabel('Expert Index')
    axes[idx].set_ylabel('Layer')
    axes[idx].set_yticks(range(len(layers)))
    axes[idx].set_yticklabels([f'Layer {l}' for l in layers])
    axes[idx].set_title(f'{label}: Expert Utilization (Step {final_ckpt})')
    plt.colorbar(im, ax=axes[idx], label='Token Share (%)')

plt.tight_layout()
plt.savefig(output_dir / 'expert_heatmap_comparison.pdf')
plt.savefig(output_dir / 'expert_heatmap_comparison.png')
plt.show()

In [ ]:
# Plot 4: Per-Layer Distribution Bar Charts (2x4 grid)
final_ckpt = stats[baseline_name]['checkpoints'][-1] if stats[baseline_name]['checkpoints'] else checkpoints[-1]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, layer in enumerate(layers):
    ax = axes[idx]
    x = np.arange(num_experts)
    width = 0.4
    
    baseline_data = all_data[baseline_name][final_ckpt]['layers'][layer]
    dro_data = all_data[dro_name][final_ckpt]['layers'][layer]
    
    if baseline_data:
        baseline_counts = np.array(baseline_data['counts'])
        baseline_pct = baseline_counts / baseline_counts.sum() * 100
        ax.bar(x - width/2, baseline_pct, width, label='Baseline', color='#2E86AB', alpha=0.7)
    
    if dro_data:
        dro_counts = np.array(dro_data['counts'])
        dro_pct = dro_counts / dro_counts.sum() * 100
        ax.bar(x + width/2, dro_pct, width, label='DRMoET', color='#A23B72', alpha=0.7)
    
    ax.axhline(100/num_experts, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Expert')
    ax.set_ylabel('Token Share (%)')
    ax.set_title(f'Layer {layer}')
    if idx == 0:
        ax.legend(loc='upper right')

plt.suptitle(f'Expert Token Distribution by Layer (Step {final_ckpt})', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(output_dir / 'per_layer_distribution.pdf', bbox_inches='tight')
plt.savefig(output_dir / 'per_layer_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# Plot 5: Aggregate Distribution Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

final_ckpt = stats[baseline_name]['checkpoints'][-1] if stats[baseline_name]['checkpoints'] else checkpoints[-1]

for idx, (model_name, label) in enumerate(models.items()):
    agg = all_data[model_name][final_ckpt]['aggregate']
    if agg:
        counts = np.array(agg['counts'])
        pct = counts / counts.sum() * 100
        
        color = '#2E86AB' if idx == 0 else '#A23B72'
        axes[idx].bar(range(num_experts), pct, color=color, alpha=0.8, edgecolor='black', linewidth=0.3)
        axes[idx].axhline(100/num_experts, color='red', linestyle='--', linewidth=2, label=f'Uniform ({100/num_experts:.2f}%)')
        
        cv = agg['cv_percent']
        axes[idx].set_xlabel('Expert Index')
        axes[idx].set_ylabel('Token Share (%)')
        axes[idx].set_title(f'{label} (CV: {cv:.2f}%)')
        axes[idx].legend()

plt.suptitle(f'Aggregate Expert Utilization (Step {final_ckpt}, All Layers)', fontsize=14)
plt.tight_layout()
plt.savefig(output_dir / 'aggregate_distribution.pdf')
plt.savefig(output_dir / 'aggregate_distribution.png')
plt.show()

In [ ]:
# Save summary stats to JSON
results = {
    'checkpoints': stats[baseline_name]['checkpoints'],
    'layers': layers,
    'num_experts': num_experts,
    'baseline': {
        'name': baseline_name,
        'cv': stats[baseline_name]['cv'],
        'gini': stats[baseline_name]['gini'],
        'per_layer_cv': {str(k): v for k, v in stats[baseline_name]['per_layer_cv'].items()},
    },
    'drmoet': {
        'name': dro_name,
        'cv': stats[dro_name]['cv'],
        'gini': stats[dro_name]['gini'],
        'per_layer_cv': {str(k): v for k, v in stats[dro_name]['per_layer_cv'].items()},
    },
    'summary': {
        'baseline_avg_cv': float(np.mean(stats[baseline_name]['cv'])) if stats[baseline_name]['cv'] else None,
        'drmoet_avg_cv': float(np.mean(stats[dro_name]['cv'])) if stats[dro_name]['cv'] else None,
    }
}

if results['summary']['baseline_avg_cv'] and results['summary']['drmoet_avg_cv']:
    results['summary']['more_uniform'] = 'Baseline' if results['summary']['baseline_avg_cv'] < results['summary']['drmoet_avg_cv'] else 'DRMoET'

with open(output_dir / 'expert_utilization_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {output_dir / 'expert_utilization_results.json'}")
print(f"Figures saved to {output_dir}/")